# **CRAWLING BERITA DARI DETIK.COM**

In [1]:
import requests
from bs4 import BeautifulSoup
import time
import re
import string
import sys
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import random

# --- FUNGSI-FUNGSI BANTUAN SCRAPING ---
def print_progress(kategori, current_page, total_pages):
    """Menampilkan progress bar di konsol."""
    percent = (current_page / total_pages) * 100 if total_pages > 0 else 0
    bar_length = 20
    filled_length = int(bar_length * current_page // total_pages) if total_pages > 0 else 0
    bar = '█' * filled_length + '-' * (bar_length - filled_length)
    sys.stdout.write(f'\r{kategori} - Page {current_page}/{total_pages} [{bar}] {percent:.2f}%')
    sys.stdout.flush()
    if current_page == total_pages:
        sys.stdout.write('\n\n')

def get_session():
    """Membuat sesi permintaan dengan mekanisme percobaan ulang."""
    session = requests.Session()
    retry_strategy = Retry(
        total=5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        backoff_factor=1
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    return session

def get_article_content_and_title(session, url):
    """Mengambil isi artikel dan judul dari URL."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36'
    }
    try:
        r = session.get(url, headers=headers, timeout=15)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        title = get_article_title(soup)
        content_selectors = [
            "div.detail-konten", "div.news-detail__content", "div.itp_bodycontent",
            "div.content-text", "div.article-content", "div.text_area"
        ]

        paragraphs = []
        for selector in content_selectors:
            content_divs = soup.select(selector)
            if content_divs:
                for div in content_divs:
                    for p in div.find_all("p"):
                        text = p.get_text(strip=True)
                        if text and not text.lower().startswith("baca juga"):
                            paragraphs.append(text)
                if paragraphs:
                    break
        
        if not paragraphs:
            body_text = soup.find("article")
            if body_text:
                for p in body_text.find_all("p"):
                    text = p.get_text(strip=True)
                    if text and not text.lower().startswith("baca juga"):
                        paragraphs.append(text)

        content = " ".join(paragraphs)
        return title, content
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}", file=sys.stderr)
        return "Judul Tidak Ditemukan", ""

def get_article_title(soup):
    """Mengambil judul artikel dari berbagai kemungkinan lokasi."""
    title_tag = soup.find("h1", class_="detail-title")
    if title_tag:
        return title_tag.get_text(strip=True)
    
    title_tag = soup.find("h2", class_="media__title")
    if title_tag:
        return title_tag.get_text(strip=True)
        
    title_tag = soup.find("title")
    if title_tag:
        return title_tag.get_text(strip=True).replace(" - detiknews", "").replace(" - detikfinance", "")

    return "Judul Tidak Ditemukan"

def extract_id(url):
    """
    Ekstrak ID berita dari URL, mengatasi format yang berbeda.
    """
    id_match_d = re.search(r"/d-(\d+)", url)
    if id_match_d:
        return id_match_d.group(1)
    id_match_end = re.search(r"-(\d+)$", url)
    if id_match_end:
        return id_match_end.group(1)
    id_match_middle = re.search(r"(\d+)\.html$", url)
    if id_match_middle:
        return id_match_middle.group(1)
        
    return None

# --- FUNGSI UTAMA SCRAPING ---
def berita(categories, pages_per_category=10):
    """Fungsi utama untuk melakukan crawling berita dan menyimpan hasilnya."""
    start_time = time.time()
    session = get_session()
    all_articles_data = []
    processed_links = set()

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36'
    }

    base_urls = {
        "politik": "https://news.detik.com/indeks/berita/",
        "hukum": "https://news.detik.com/indeks/berita/",
        "ekonomi": "https://finance.detik.com/indeks/",
        "detikx": "https://news.detik.com/x/indeks/",
        "hiburan": "https://hot.detik.com/indeks/",
        "internasional": "https://news.detik.com/indeks/berita/",
        "sepakbola": "https://sport.detik.com/sepakbola/indeks/",
        "olahraga": "https://sport.detik.com/indeks/",
        "lingkungan": "https://www.detik.com/tag/lingkungan",
        "otomotif": "https://oto.detik.com/indeks"
    }
    
    categories = list(set(categories))

    for cat in categories:
        current_url = base_urls.get(cat.lower(), f"https://{cat.lower()}.detik.com/indeks/")
        print(f"--- Memulai crawling untuk kategori: {cat} ---")
        
        for page_count in range(1, pages_per_category + 1):
            url = f"{current_url}?page={page_count}"
            if cat.lower() == "lingkungan":
                url = f"https://www.detik.com/tag/lingkungan?page={page_count}"
            
            print_progress(cat, page_count, pages_per_category)

            try:
                r = session.get(url, headers=headers, timeout=15)
                r.raise_for_status()
                soup = BeautifulSoup(r.text, "html.parser")
                article_links = soup.select("a.media__link")
                
                for a in article_links:
                    link = a["href"]
                    if link in processed_links:
                        continue
                    processed_links.add(link)
                    berita_id = extract_id(link)
                    title, content = get_article_content_and_title(session, link)
                    
                    if content:
                        print(f"\n--- Data {len(all_articles_data) + 1} ---")
                        print(f"ID Berita: {berita_id}")
                        print(f"Judul: {title}")
                        print(f"Abstrak (Raw): {content}")
                        print(f"Kategori: {cat}")
                        print("-------------------------------------------\n")

                        all_articles_data.append({
                            "id_berita": berita_id,
                            "judul_berita": title,
                            "isi_berita_original": content,
                            "kategori_berita": cat
                        })
                    time.sleep(random.uniform(1, 3))
            except requests.exceptions.RequestException as e:
                print(f"\n❌ Gagal mengakses {url}: {e}", file=sys.stderr)
                break
    
    df = pd.DataFrame(all_articles_data)
    df.to_csv("crawling_detik_berita.csv", index=False, encoding="utf-8-sig")

    end_time = time.time()
    elapsed = int(end_time - start_time)
    jam, sisa = divmod(elapsed, 3600)
    menit, detik = divmod(sisa, 60)

    print("\n✅ Seluruh data berhasil dikumpulkan!")
    print(f"📊 Total entri: {len(df)}")
    print(f"⏱️ Waktu eksekusi: {jam} jam {menit} menit {detik} detik")
    
    print("\nBerikut adalah 5 entri pertama yang berhasil dikumpulkan:")
    print(df.head())

    return df

if __name__ == '__main__':
    categories = ["politik", "hukum", "ekonomi", "lingkungan", "hiburan", "internasional", "otomotif", "olahraga", "sepakbola"]
    berita(categories, pages_per_category=5)

--- Memulai crawling untuk kategori: otomotif ---
otomotif - Page 1/5 [████----------------] 20.00%


--- Data 1 ---
ID Berita: 8666291
Judul: Pajak Mercy Klasik Purbaya, Cuma Bayar Segini Setiap Tahun
Abstrak (Raw): Purbaya Yudhi Sadewa membagikan momen menarik saat masih menjabat Menteri Keuangan. Purbaya mengendarai sendiri Mercedes-Benz lawas dari rumah menuju kantor Kementerian Keuangan. Mobil tersebut merupakan sedan Mercedes-Benz C200K keluaran 2008. Berapa ya pajak tahunannya? Dalam unggahannya, Purbaya mengatakan mobil tersebut sudah lama tidak digunakan. Dia mengenang kendaraan itu sebagai mobil yang pernah dipakainya saat bekerja di Danareksa. "Udah lama ga pake Mercy Lama. Waktu kerja di Danarekasa. Awet, masih ada. Masih bagus dan ganteng," tulis Purbaya di unggahan TikTok-nya. Mobil berkelir abu-abu metalik tersebut pakai pelat nomor B 8529 ZH. Berdasarkan data yang ditelusuri di laman Samsat DKI Jakarta, Mercedes-Benz C200K itu mempunyai kapasitas mesin 1.796 cc dengan nilai jual kendaraan Rp 128 juta. Pajak tahunannya tercatat sebesar Rp 2.767.000, dengan status pajak 


--- Data 2 ---
ID Berita: 8666263
Judul: Benarkah Ada Razia Pajak Kendaraan dari Rumah ke Rumah?
Abstrak (Raw): Beredar informasi yang menyebut ada razia pajak kendaraan dari rumah ke rumah. Benarkah demikian? Muncul poster yang menyebutkan akan ada razia pajak kendaraan yang dilakukan oleh pihak kepolisian dan juga asosiasi perusahaan pembiayaan. Dalam poster yang beredar, tim gabungan polisi dan perusahaan pembiayaan akan mendatangi rumah-rumah untuk melakukan penertiban dan pemeriksaaan kendaraan bermotor pada akhir September 2026. SCROLL TO CONTINUE WITH CONTENT Tim gabungan disebut akan melakukan pemeriksaan terhadap kendaraan dengan pajak mati, kendaraan dengan hanya kepemilikan STNK, kendaraan sengketa kredit, dan kendaraan over kredit ilegal. Masyarakat juga diimbau melakukan pelunasan pajak kendaraan sesegera mungkin. Dokumen kendaraan juga diminta lengkap dan menyelesaikan tunggakan kredit sebelum pemeriksaan. Akun Instagram Korlantas Polri menegaskan bahwa poster sekaligus 


--- Data 3 ---
ID Berita: 8666256
Judul: SIM Palsu Dijual Rp 550 Ribu, Bikin yang Asli Cuma Segini
Abstrak (Raw): Polisi membongkar praktik pembuatan Surat Izin Mengemudi (SIM) palsu. Harga yang ditawarkan mulai dari Rp 550 ribu. Sebenarnya berapa duit yang perlu dipersiapkan jika jalur resmi? Diberitakan detikNews sebelumnya, pengungkapan bermula saat polisi menangkap seorang tersangka ketika menyerahkan SIM di Desa Suka Mulia, Kecamatan Dayun, Siak. Dari penangkapan itu, penyidik melakukan pengembangan hingga berhasil mengamankan tersangka lainnya. "Total delapan orang telah kami tetapkan sebagai tersangka dan dua lainnya masih dalam pencarian. Kami terus mengembangkan penyidikan untuk mengetahui luas peredaran SIM palsu serta pihak lain yang terlibat," ujar Kapolres Siak AKBP Sepuh Ade Irsyam Siregar. SCROLL TO CONTINUE WITH CONTENT Polisi menyebut para pelaku menawarkan pembuatan SIM melalui WhatsApp, marketplace Facebook, dan perantara dengan tarif Rp 550 ribu hingga Rp 1,7 juta.


--- Data 4 ---
ID Berita: 8665702
Judul: Cat Mobil Lokal Naik Kelas, Siap Bersaing dengan Produk Luar Negeri
Abstrak (Raw): Industri cat mobil lokal naik kelas. Menjanjikan kualitas yang bagus, cat lokal siap bertarung dengan produk dari luar negeri. Bicara industri otomotif Indonesia, bukan cuma urusan penjualan mobil baru. Di balik semakin banyaknya mobil yang beredar di jalan, bisnis pendukungnya pun ikut berkembang. Salah satunya industri cat dan finishing kendaraan. Terlebih di era banyak mobil baru bermunculan, kebutuhan akan body repair, pengecatan ulang, hingga modifikasi seolah menjadi kebutuhan baru. SCROLL TO CONTINUE WITH CONTENT Body repair, pengecatan ulang merupakan upaya penting untuk menjaga tampilan mobil. Maka dari itu, kualitas hasil akhir sudah pasti jadi sorotan. Peluang tersebut turut mendorong pelaku industri cat dalam negeri untuk mengembangkan produk yang mampu memenuhi kebutuhan pasar sekaligus bersaing dengan produk dari luar negeri. Hal itulah yang tengah 


--- Data 5 ---
ID Berita: 8665601
Judul: Larisnya SUV Off Road Geely, Sejam Terpesan 43 Ribu Unit!
Abstrak (Raw): Geely Galaxy resmi membuka pre-order SUV off-road pertamanya, Galaxy Zhanjian 700. Dalam sembilan menit, pemesanannya langsung menembus 10.000 unit. Dalam satu jam, refundable order Zhanjian 700 mencapai lebih dari 43.900 unit. Edisi terbatas Land Ark yang hanya tersedia 700 unit juga langsung ludes. SCROLL TO CONTINUE WITH CONTENT Zhanjian 700 ditawarkan dalam enam varian dengan harga pre-order 199.800-389.800 yuan atau sekira Rp 520jutaan hingga Rp 1,02 miliaran. SUV ini mengusung desain "Oriental Lion" dengan lampu Sky Eye, DRL memanjang, dan logo Geely yang menyala. Dimensinya mencapai 5.085 x 1.999 x 1.895 mm dengan wheelbase 2.900 mm. Ground clearance-nya 233 mm, approach angle 30 derajat, departure angle 31 derajat, dan kemampuan menerjang air hingga 800 mm. Zhanjian 700 menggunakan sistem hybrid Thor EM-T AWD. Pilihan mesinnya 1.5T 120 kW dan 2.0T 160 kW, yang dipa


--- Data 6 ---
ID Berita: 8665617
Judul: Viral Mobil PHEV Terbakar, Dealer Klaim gara-gara Powerbank
Abstrak (Raw): Sebuah mobil SUV plug-in hybrid (PHEV) Lynk & Co 900 terbakar di tepi jalan di Hangzhou, China baru-baru ini. Dealer lokal mengklaim kebakaran terjadi karena powerbank. Dikutip media lokal Carnewschina, dealer lokal merek tersebut menepis spekulasi bahwa SUV plug-in hybrid itu mengalami kebakaran akibat baterai tegangan tinggi. Pihak dealer menyatakan bahwa sebuah powerbank milik konsumen dilaporkan terjatuh dan tertendang ke bawah mobil sebelum api berkobar. Meski begitu, penyebab pastinya masih menunggu hasil penyelidikan pemadam kebakaran setempat. Insiden tersebut terjadi di sebuah kawasan komersial di Hangzhou, Provinsi Zhejiang, China, pada tanggal 9 September 2026 sekitar pukul 16.06. Rekaman video dari saksi mata memperlihatkan kepulan asap dan kobaran api di bagian bawah SUV yang sedang terparkir sebelum petugas pemadam kebakaran memadamkan api tersebut. Rekaman


--- Data 7 ---
ID Berita: 8665726
Judul: Duh! Sigra Pakai Bemper Berduri dari Besi, Kayak Mobil Perang
Abstrak (Raw): Daihatsu Sigra keciduk menggunakan bemper tajam dari besi. Sigra itu juga kedapatan pakai pelat nomor dengan lis biru yang diperuntukkan buat mobil listrik. Anggota Satlantas Polrestabes Makassar Mahir Daeng Rani menyetop Daihatsu Sigra yang menggunakan aksesoris membahayakan. Dalam video yang diunggah akun Instagram mahirwttdaeng, Daihatsu Sigra berkelir putih itu menggunakan aksesoris di bemper berupa pelat besi berduri yang cukup tajam. SCROLL TO CONTINUE WITH CONTENT [Gambas:Instagram] "Nggak boleh ini, bahaya ini, kayak mobil perang," demikian ungkap petugas kepolisian dalam video. "Katanya pengaman ini, runcing sekali. Kalau ada yang tabrak belakang ini bisa cedera parah," tambahnya lagi. Petugas kemudian meminta SIM dan STNK pengemudi yang bersangkutan. Dia menegaskan bahwa pemasangan aksesoris tambahan bemper diperbolehkan namun tidak membahayakan pengguna jala


--- Data 8 ---
ID Berita: 8665579
Judul: Robot Manusia Buatan Chery Unjuk Gigi di Eropa: Jadi Perawat-Petugas Keamanan
Abstrak (Raw): AiMOGA Robotics, anak perusahaan Chery unjuk gigi dalam ajang Innovation for All (IFA) 2026 di Berlin, Jerman. Dalam pameran tersebut, AiMOGA menampilkan Robot Keamanan Cerdas (Intelligent Security Robot), Robot Perawat (Robotic Nurse), robot humanoid Mornine, serta robot quadruped Argos. Debut internasional ini bukan sekadar pameran produk. AiMOGA menyebut Intelligent Security Robo telah digunakan di lebih dari 30 kota di Tiongkok untuk mendukung berbagai tugas nyata, seperti layanan keamanan dan pengaturan ketertiban. Sementara itu, Robotic Nurse berfokus pada kebutuhan layanan kesehatan. Penerapannya mencakup pengarahan dan konsultasi pasien, bantuan informasi, serta layanan di lingkungan rumah sakit. SCROLL TO CONTINUE WITH CONTENT Selain kedua robot layanan profesional tersebut, Mornine turut tampil dalam program resmi IFA bertajuk "Robots on the R

In [4]:
import re
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# Gunakan data hasil crawling yang sudah ada di DataFrame.
if "df" not in globals():
    df = pd.read_csv("crawling_detik_berita.csv")

kolom_teks = "isi_berita_original"
if kolom_teks not in df.columns:
    raise KeyError(f"Kolom '{kolom_teks}' tidak ditemukan. Kolom yang tersedia: {list(df.columns)}")

teks_berita = df[kolom_teks].fillna("").astype(str)

# Ambil kata alfabet, ubah menjadi huruf kecil, dan abaikan kata satu huruf.
vectorizer = CountVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b[a-zA-ZÀ-ÿ]{2,}\b"
)
matriks_kata = vectorizer.fit_transform(teks_berita)
kata_unik = vectorizer.get_feature_names_out()

print(f"Jumlah berita yang dianalisis: {len(teks_berita)}")
print(f"Jumlah kata unik: {len(kata_unik)}")
print("100 kata unik pertama:", list(kata_unik[:100]))

Jumlah berita yang dianalisis: 587
Jumlah kata unik: 15680
100 kata unik pertama: ['aam', 'aamiin', 'aan', 'aarhus', 'aau', 'abad', 'abadi', 'abah', 'abal', 'abang', 'abd', 'abdi', 'abdul', 'abdullah', 'abdulrachman', 'aberdeen', 'abidal', 'abis', 'abk', 'abraham', 'abrasi', 'abrasif', 'abs', 'absen', 'absennya', 'absolument', 'abu', 'ac', 'acara', 'accurate', 'aceh', 'achmad', 'achraf', 'acid', 'acl', 'acosta', 'acostabelum', 'acostamulai', 'activ', 'active', 'actuation', 'acuan', 'acuannya', 'ada', 'adakan', 'adalah', 'adalahalwi', 'adalahbruno', 'adalahgunawan', 'adalahjonatan', 'adalahlionel', 'adalahsocial', 'adam', 'adanya', 'adaptabilitas', 'adaptasi', 'adaptif', 'adaptive', 'adapun', 'adas', 'adasadio', 'adat', 'adatanda', 'adegan', 'adelaide', 'adem', 'adeyemi', 'adham', 'adhfar', 'adhi', 'adhitya', 'adi', 'adianto', 'adik', 'adiknya', 'adil', 'adininggar', 'aditia', 'aditya', 'adjie', 'adk', 'adm', 'admin', 'administrasi', 'administrasinya', 'administratif', 'administratifnya